<div style="background:linear-gradient(135deg,#3b1d6e 0%,#6d28d9 55%,#a855f7 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#e6dbff;font-weight:700;text-transform:uppercase">Chapter 29 · Solutions</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">Penguin Challenges, Worked Answers ✅</div>
  <div style="font-size:15px;color:#efe7ff;max-width:720px;line-height:1.6">Full solutions to the five Palmer penguins challenges. Try them yourself first, then compare.</div>
  <div style="margin-top:16px;font-size:13px;color:#e6dbff">Statistics, Data Science and AI: A Visual Handbook · John Fisher · 2026</div>
</div>

## ⚙️ Setup

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
CYAN="#0891b2"; AMBER="#d97706"; PURPLE="#7c3aed"; INK="#1a2138"; GRID="#e6e9f2"
SPC={"Adelie":CYAN,"Chinstrap":AMBER,"Gentoo":PURPLE}
plt.rcParams.update({"figure.dpi":110,"axes.grid":True,"grid.color":GRID,"axes.axisbelow":True,
   "axes.spines.top":False,"axes.spines.right":False,"axes.titleweight":"bold"})
BASE="https://raw.githubusercontent.com/johnfisher-ai/Statistics-Data-Science-AI-Visual-Book/main/data/"
try:    df = pd.read_csv("../../data/penguins.csv")
except FileNotFoundError: df = pd.read_csv(BASE+"penguins.csv")
print("loaded:", df.shape)

loaded: (347, 10)


<div style="background:#f3edff;border-left:5px solid #7c3aed;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#7c3aed;letter-spacing:1px">CHALLENGE 1 · UNIT ERROR</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Fix, do not delete</div>
<div style="color:#4a5578;margin-top:6px">One flipper_length_mm reads 19.5 while the rest are near 195. Show it is a unit error, correct it, and explain why you fix rather than delete it.</div>
</div>

In [2]:
d = df.drop_duplicates().copy()
print("suspicious flippers (<50mm):", d.loc[d["flipper_length_mm"]<50,"flipper_length_mm"].to_list())
d.loc[d["flipper_length_mm"]<50, "flipper_length_mm"] *= 10
print("flipper range after fix:", d["flipper_length_mm"].min(), "to", d["flipper_length_mm"].max())

suspicious flippers (<50mm): [19.5]
flipper range after fix: 165.0 to 237.0


**Answer:** 19.5 is impossible for a penguin flipper (~190 mm), and it is exactly one-tenth of a plausible value, the tell-tale sign of **centimetres recorded as millimetres**. Because we understand the cause, we *correct* it (×10) rather than deleting a real bird's data. An outlier with a known, fixable cause is repaired, not discarded (Chapter 21).

<div style="background:#f3edff;border-left:5px solid #7c3aed;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#059669;letter-spacing:1px">CHALLENGE 2 · DROP EMPTY ROWS</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">No measurements, no value</div>
<div style="color:#4a5578;margin-top:6px">Two rows are missing every measurement. Drop only those rows (not rows missing just one value) and report the count before and after.</div>
</div>

In [3]:
meas = ["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]
before = len(d)
d = d.dropna(subset=meas, how="all")
print(f"rows: {before} -> {len(d)}  (dropped {before-len(d)} all-missing rows)")
print("rows still missing at least one measurement:", int(d[meas].isna().any(axis=1).sum()))

rows: 344 -> 342  (dropped 2 all-missing rows)
rows still missing at least one measurement: 0


**Answer:** `dropna(subset=meas, how="all")` removes only the rows where *every* measurement is missing, the ones that carry no information. It deliberately keeps rows missing just one value, which can still be imputed or used. Matching the rule to the situation (`how="all"` vs the default `how="any"`) is the careful move (Chapter 20).

<div style="background:#f3edff;border-left:5px solid #7c3aed;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#d97706;letter-spacing:1px">CHALLENGE 3 · CLEAN &amp; IMPUTE SEX</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Standardise, then fill smartly</div>
<div style="color:#4a5578;margin-top:6px">The sex column has casing issues, '.' and blanks, and some true gaps. Standardise it, then impute the gaps with the mode within each species. Why within species?</div>
</div>

In [4]:
d["sex"] = d["sex"].astype("string").str.strip().str.lower().replace({".":pd.NA,"":pd.NA})
print("after standardising:", d["sex"].value_counts(dropna=False).to_dict())
n = int(d["sex"].isna().sum())
d["sex"] = d.groupby("species")["sex"].transform(lambda s: s.fillna(s.mode().iloc[0]))
print(f"imputed {n} gaps within species:", d["sex"].value_counts().to_dict())

after standardising: {'male': 176, 'female': 157, <NA>: 9}
imputed 9 gaps within species: {'male': 183, 'female': 159}


**Answer:** First standardise (trim, lowercase, map `"."` and blanks to `NaN`) so the real categories are just `male`/`female` (Chapter 19). Then impute the few remaining gaps with the most common sex **within the same species**, because the species differ in size and sex ratio, so a species-aware fill is more defensible than a single global mode (Chapter 20). Clean before you impute, always.

<div style="background:#f3edff;border-left:5px solid #7c3aed;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#2563eb;letter-spacing:1px">CHALLENGE 4 · SIMPSON'S PARADOX</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">The correlation that lies</div>
<div style="color:#4a5578;margin-top:6px">Compute the correlation between bill_length and bill_depth overall, then within each species. Explain the reversal and what it teaches.</div>
</div>

In [5]:
sub = d.dropna(subset=["bill_length_mm","bill_depth_mm"])
print("overall r:", round(sub["bill_length_mm"].corr(sub["bill_depth_mm"]),2))
for sp in SPC:
    s = sub[sub["species"]==sp]
    print(f"  within {sp:<10}: r =", round(s["bill_length_mm"].corr(s["bill_depth_mm"]),2))

overall r: -0.29
  within Adelie    : r = 0.28
  within Chinstrap : r = 0.23
  within Gentoo    : r = 0.3


**Answer:** Overall the correlation is **negative** (≈ −0.29), but **within every species it is positive** (≈ +0.2 to +0.3). This is **Simpson's paradox**: a hidden grouping variable (species) reverses the apparent relationship. Gentoos have long, shallow bills and form a separate cluster, so pooling all species creates a misleading downward trend. The lesson is permanent: a correlation across mixed groups can lie, so always check whether a lurking category explains it (Chapters 16 and correlation-vs-causation).

<div style="background:#f3edff;border-left:5px solid #7c3aed;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#7c3aed;letter-spacing:1px">CHALLENGE 5 · ENCODE BY TYPE</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Ordinal vs one-hot, and a no-signal check</div>
<div style="color:#4a5578;margin-top:6px">Ordinal-encode body_size and one-hot encode band_color. Then test whether band_color predicts body mass. What do you conclude?</div>
</div>

In [6]:
d["body_size"]  = d["body_size"].astype("string").str.strip().str.lower().replace({"med":"medium"})
d["band_color"] = d["band_color"].astype("string").str.strip().str.lower()
d["body_size_code"] = d["body_size"].map({"small":1,"medium":2,"large":3})   # ORDINAL
band_oh = pd.get_dummies(d["band_color"], prefix="band")                      # NOMINAL
print("body_size_code values:", sorted(d["body_size_code"].dropna().unique()))
print("one-hot band columns:", list(band_oh.columns))
print("\nmean body mass by band_color:")
print(d.groupby("band_color")["body_mass_g"].mean().round(0).to_string())

body_size_code values: [np.int64(1), np.int64(2), np.int64(3)]
one-hot band columns: ['band_blue', 'band_green', 'band_red', 'band_yellow']

mean body mass by band_color:
band_color
blue      4224.0
green     4239.0
red       4206.0
yellow    4213.0


**Answer:** `body_size` has a real order (small &lt; medium &lt; large), so it becomes integer codes **1/2/3**. `band_color` has no order, so it becomes **one-hot** 1/0 columns, label-encoding it (red=1, blue=2…) would invent a ranking that does not exist (Chapter 24). The no-signal check seals it: mean body mass is ~4,200 g for *every* colour, so the research tag carries no information. Proving a variable is noise is a real finding, it tells you not to waste a model on it.

---
<div style="background:#ffffff;border:1px solid #e6e9f2;border-radius:16px;padding:22px 26px;font-family:Inter,sans-serif">
<div style="font-size:19px;font-weight:800;color:#1a2138">🎉 Part V complete</div>
<div style="color:#4a5578;line-height:1.8;margin-top:8px">You fixed a unit error, dropped empty rows, cleaned and imputed a category, exposed Simpson's paradox, and encoded by type. Three datasets, one repeatable routine, and the whole first half of the book put to work.</div>
</div>

<div style="text-align:center;color:#8b94b3;font-size:12px;margin-top:16px">Statistics, Data Science and AI: A Visual Handbook · © 2026 John Fisher</div>